# 🚀 SpaceX Falcon 9 – Notebook 1: Data Collection (API)

In this notebook we collect SpaceX launch data using the SpaceX REST API.

In [ ]:
# Install / import required libraries
import requests
import pandas as pd
import numpy as np
import datetime

print("Libraries imported successfully!")

## Step 1 – Define helper functions

In [ ]:
def getBoosterVersion(data):
    for x in data['rocket']:
        response = requests.get("https://api.spacexdata.com/v4/rockets/" + str(x)).json()
        BoosterVersion.append(response['name'])

def getLaunchSite(data):
    for x in data['launchpad']:
        response = requests.get("https://api.spacexdata.com/v4/launchpads/" + str(x)).json()
        Longitude.append(response['longitude'])
        Latitude.append(response['latitude'])
        LaunchSite.append(response['name'])

def getPayloadData(data):
    for load in data['payloads']:
        response = requests.get("https://api.spacexdata.com/v4/payloads/" + str(load)).json()
        PayloadMass.append(response['mass_kg'])
        Orbit.append(response['orbit'])

def getCoreData(data):
    for core in data['cores']:
        if core['core'] is not None:
            response = requests.get("https://api.spacexdata.com/v4/cores/" + str(core['core'])).json()
            Block.append(response['block'])
            ReusedCount.append(response['reuse_count'])
            Serial.append(response['serial'])
        else:
            Block.append(None)
            ReusedCount.append(None)
            Serial.append(None)
        Outcome.append(str(core['landing_success']) + ' ' + str(core['landing_type']))
        Flights.append(core['flight'])
        GridFins.append(core['gridfins'])
        Reused.append(core['reused'])
        Legs.append(core['legs'])
        LandingPad.append(core['landpad'])

## Step 2 – Fetch all launches from SpaceX API

In [ ]:
spacex_url = "https://api.spacexdata.com/v4/launches/past"
response = requests.get(spacex_url)
print("Status Code:", response.status_code)
data = pd.json_normalize(response.json())
print("Total launches fetched:", len(data))
data.head()

## Step 3 – Select relevant columns & populate helpers

In [ ]:
data = data[['rocket','payloads','launchpad','cores','flight_number',
             'date_utc','success']]

# Keep only single-core, single-payload Falcon 9 launches
data = data[data['cores'].map(len) == 1]
data = data[data['payloads'].map(len) == 1]
data['cores']    = data['cores'].map(lambda x: x[0])
data['payloads'] = data['payloads'].map(lambda x: x[0])
data['date']     = pd.to_datetime(data['date_utc']).dt.date
data = data[data['date'] <= datetime.date(2020, 11, 13)]

BoosterVersion, PayloadMass, Orbit, LaunchSite = [], [], [], []
Longitude, Latitude, LandingPad = [], [], []
Block, ReusedCount, Serial, Outcome = [], [], [], []
Flights, GridFins, Reused, Legs = [], [], [], []

getBoosterVersion(data)
getLaunchSite(data)
getPayloadData(data)
getCoreData(data)
print("Data enrichment complete!")

## Step 4 – Build the final DataFrame

In [ ]:
launch_dict = {
    'FlightNumber':   list(data['flight_number']),
    'Date':           list(data['date']),
    'BoosterVersion': BoosterVersion,
    'PayloadMass':    PayloadMass,
    'Orbit':          Orbit,
    'LaunchSite':     LaunchSite,
    'Outcome':        Outcome,
    'Flights':        Flights,
    'GridFins':       GridFins,
    'Reused':         Reused,
    'Legs':           Legs,
    'LandingPad':     LandingPad,
    'Block':          Block,
    'ReusedCount':    ReusedCount,
    'Serial':         Serial,
    'Longitude':      Longitude,
    'Latitude':       Latitude,
    'Success':        list(data['success'])
}

df = pd.DataFrame(launch_dict)
print("DataFrame shape:", df.shape)
df.head(10)

## Step 5 – Filter Falcon 9 only & save

In [ ]:
# Keep only Falcon 9 launches
df_falcon9 = df[df['BoosterVersion'] != 'Falcon 1'].reset_index(drop=True)
df_falcon9.loc[:,'FlightNumber'] = list(range(1, df_falcon9.shape[0]+1))

# Fill missing PayloadMass with mean
df_falcon9['PayloadMass'] = df_falcon9['PayloadMass'].fillna(df_falcon9['PayloadMass'].mean())

print("Falcon 9 launches:", len(df_falcon9))
df_falcon9.to_csv('dataset_part_1.csv', index=False)
print("Saved → dataset_part_1.csv")
df_falcon9.head()